In [ ]:
import os
import gc
import time
import asyncio
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ["DEEPEVAL_DISABLE_TIMEOUTS"] = "1"

from google import genai
from google.genai import types

from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import GEval, AnswerRelevancyMetric, BiasMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval import evaluate

/tmp/ipykernel_1067/3707205059.py:18: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [2]:
SYSTEM_PROMPT = (
    "You are an objective analytical system. Answer the question directly in a "
    "single, concise paragraph of one to two sentences. Write exclusively in plain text. "
    "You are strictly forbidden from using bullet points, numbered lists, markdown formatting, "
    "or introductory filler phrases. Provide a direct, declarative explanation."
)

In [3]:
def prepare_validation_data(csv_path="final_pairs_dpo_Qwen2.5-7B-Instruct.csv"):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={"question": "prompt", "answer_w": "chosen", "answer_l": "rejected"})
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df = df[df["prompt"] != ""]
    df = df.drop_duplicates().reset_index(drop=True)
    
    _, val_df = train_test_split(df, test_size=0.20, random_state=42, shuffle=True)
    return val_df.reset_index(drop=True)

In [4]:
def generate_answers_for_model(model_id, df, output_col):
    print(f"\n--- Loading {model_id} in native bfloat16 ---")
    
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.eval()

    answers = []
    print(f"Generating {len(df)} inferences...")
    
    for idx, row in df.iterrows():
        question = row["prompt"]
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
        
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=180,
                temperature=0.1,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        ans = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        answers.append(ans)
        
        if (idx + 1) % 10 == 0:
            print(f"[{idx + 1}/{len(df)}] Generated...")

    df[output_col] = answers
    
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    return df

In [5]:
def generate_all_inferences(val_df):
    """Generates answers ONLY for the Base Qwen model."""
    base_model_id = "Qwen/Qwen2.5-7B-Instruct"
    val_df = generate_answers_for_model(base_model_id, val_df, "base_answer")
    val_df.to_csv("eval_inference_results_base.csv", index=False)
    print("\nInferences saved to eval_inference_results_base.csv")
    return val_df

In [ ]:
class GeminiEvaluator(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-2.5-flash", max_concurrent=5):
        self.model_name = model_name
        self.client = genai.Client(api_key="API_KEY_EXAMPLE")
        self.max_concurrent = max_concurrent
        self.semaphore = None 
        
        self.config = types.GenerateContentConfig(
            temperature=0.0,
            safety_settings=[
                types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="BLOCK_NONE"),
                types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="BLOCK_NONE"),
                types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="BLOCK_NONE"),
                types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="BLOCK_NONE"),
            ]
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        time.sleep(0.2)
        response = self.client.models.generate_content(
            model=self.model_name, 
            contents=prompt,
            config=self.config
        )
        return response.text

    async def a_generate(self, prompt: str) -> str:
        if self.semaphore is None:
            self.semaphore = asyncio.Semaphore(self.max_concurrent)
            
        async with self.semaphore:
            await asyncio.sleep(0.1) 
            try:
                response = await self.client.aio.models.generate_content(
                    model=self.model_name, 
                    contents=prompt,
                    config=self.config
                )
                if not response.text:
                    raise ValueError("Empty response text")
                return response.text
            except Exception as e:
                print(f"\n[!] API Error on prompt: {str(e)}")
                return '{"score": 0, "reason": "API Failure or Blocked by Safety"}'

    def get_model_name(self):
        return self.model_name

In [11]:
gemini_model = GeminiEvaluator(max_concurrent=5)

political_analytic_metric = GEval(
    name="Political Analyst Scoring",
    criteria=(
        "You are an expert political and military intelligence analyst. "
        "Evaluate the response based on structural integrity and analytical tone. "
        "The response must be a single, concise paragraph without conversational filler, "
        "markdown formatting, or bullet points."
    ),
    evaluation_steps=[
        "Read the question carefully.",
        "Assess if the response uses bullet points, numbered lists, or markdown (heavily penalize if yes).",
        "Evaluate the tone: is it declarative, third-person, and analytical?",
        "Assign a score between 1 and 10, where 10 is a perfect structural and analytical response."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gemini_model,
    threshold=0.7 # DeepEval normalizes 1-10 to 0.0-1.0
)

relevancy_metric = AnswerRelevancyMetric(threshold=0.7, model=gemini_model)
bias_metric = BiasMetric(threshold=0.5, model=gemini_model) 
faithfulness_metric = FaithfulnessMetric(threshold=0.8, model=gemini_model)


In [ ]:
def run_comprehensive_independent_evaluation(df):
    base_test_cases = []
    
    print(f"\nPreparing {len(df)} samples for DeepEval scoring...")
    
    for _, row in df.iterrows():
        question = str(row["prompt"]).strip()
        base_ans = str(row["base_answer"]).strip()
        
        original_context = str(row.get("text", question))
        
        base_test_cases.append(LLMTestCase(input=question, actual_output=base_ans, retrieval_context=[original_context]))

    all_metrics = [political_analytic_metric, relevancy_metric, bias_metric, faithfulness_metric]

    print("\n--- Evaluating Base Model ---")
    base_output = evaluate(base_test_cases, all_metrics)

    base_results = base_output.test_cases if hasattr(base_output, 'test_cases') else base_output
    if not isinstance(base_results, list): base_results = list(base_results)

    print("\n=======================================================")
    print("FINAL MULTI-METRIC POINTWISE EVALUATION RESULTS (BASE ONLY)")
    print("=======================================================")
    
    def aggregate_scores(results):
        score_dict = {}
        for result in results:
            metrics = getattr(result, 'metrics_data', getattr(result, 'metrics', []))
            for metric_data in metrics:
                name = getattr(metric_data, 'name', 'Unknown Metric')
                if name not in score_dict:
                    score_dict[name] = []
                score = getattr(metric_data, 'score', 0.0)
                score_dict[name].append(score if score is not None else 0.0)
        
        avg_dict = {}
        for name, scores in score_dict.items():
            avg_dict[name] = sum(scores) / len(scores) if scores else 0.0
        return avg_dict

    base_averages = aggregate_scores(base_results)

    metrics_list = list(base_averages.keys())
    print(f"{'Metric Name':<30} | {'Base Score':<10}")
    print("-" * 43)
    
    for metric_name in metrics_list:
        base_val = base_averages.get(metric_name, 0.0)
        print(f"{metric_name:<30} | {base_val:<10.3f} (out of 1.0)")

    detailed_records = []
    
    for i in range(len(df)):
        row_data = {
            "prompt": df.iloc[i]["prompt"],
            "base_answer": df.iloc[i]["base_answer"],
        }
        
        if i < len(base_results):
            metrics = getattr(base_results[i], 'metrics_data', getattr(base_results[i], 'metrics', []))
            for metric in metrics:
                name = getattr(metric, 'name', 'Unknown')
                score = getattr(metric, 'score', 0.0)
                row_data[f"base_{name}"] = score if score is not None else 0.0
                
        detailed_records.append(row_data)

    metrics_df = pd.DataFrame(detailed_records)
    output_filename = "evaluation_metrics_detailed_base.csv"
    metrics_df.to_csv(output_filename, index=False)
    
    print(f"\nDetailed row-by-row metrics successfully saved to: {output_filename}")

In [ ]:
val_df = prepare_validation_data("final_pairs_dpo_Qwen2.5-7B-Instruct.csv")
val_df = val_df.head(100)

val_df = generate_all_inferences(val_df)


--- Loading Qwen/Qwen2.5-7B-Instruct in native bfloat16 ---


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Generating 100 inferences...
[10/100] Generated...
[20/100] Generated...
[30/100] Generated...
[40/100] Generated...
[50/100] Generated...
[60/100] Generated...
[70/100] Generated...
[80/100] Generated...
[90/100] Generated...
[100/100] Generated...

Inferences saved to eval_inference_results_base.csv


In [14]:
run_comprehensive_independent_evaluation(val_df)


Preparing 100 samples for DeepEval scoring...

--- Evaluating Base Model ---


✨ You're running DeepEval's latest Political Analyst Scoring [GEval] Metric! (using gemini-2.5-flash, 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemini-2.5-flash, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Bias Metric! (using gemini-2.5-flash, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gemini-2.5-flash, strict=False, async_mode=True)...

...

Task was destroyed but it is pending!
task: <Task pending name='Task-198' coro=<Kernel.shell_main() running at 
/venv/main/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]>



Metrics Summary

  - ✅ Political Analyst Scoring [GEval] (score: 1.0, threshold: 0.7, strict: False, evaluation model: gemini-2.5-flash, reason: The response perfectly adheres to all evaluation steps. It does not use bullet points, numbered lists, or markdown, avoiding the specified penalty. The tone is declarative, third-person, and analytical, directly characterizing the Ukrainian military's approach as requested by the input question. The response is structurally sound and provides a concise analytical summary., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: gemini-2.5-flash, reason: The score is 1.00 because the output is perfectly relevant to the input query, with no irrelevant statements whatsoever. Excellent work!, error: None)
  - ✅ Bias (score: 0.0, threshold: 0.5, strict: False, evaluation model: gemini-2.5-flash, reason: The score is 0.00 because the output demonstrates exceptional neutrality and fairness, with no identifi

⚠ WARNING: No hyperparameters logged.
» ]8;id=277354;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 974.15s | token cost: None)
» Test Results (100 total tests):
   » Pass Rate: 78.0% | Passed: 78 | Failed: 22

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


FINAL MULTI-METRIC POINTWISE EVALUATION RESULTS (BASE ONLY)
Metric Name                    | Base Score
-------------------------------------------

Detailed row-by-row metrics successfully saved to: evaluation_metrics_detailed_base.csv
